# Base model (Pretrained)

In [ ]:
# ============================================================
# EVAL MÔ HÌNH THÔ (BASE MODEL ONLY - NO LORA - NO CONTEXT)
# ============================================================
import json
import os
import torch
from tqdm import tqdm
import evaluate
import warnings
from bert_score import score as bert_score

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Ép chạy 1 GPU để tránh nghẽn mạch Multi-GPU khi sinh chữ

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ------------------------------------------------------------
# 1. Cấu hình đường dẫn và tham số
# ------------------------------------------------------------
BASE_MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
TEST_PATH      = "/kaggle/input/datasets/chichidanghocbai/qa-rag-vietamculture/final_test.jsonl"
OUTPUT_PATH    = "/kaggle/working/eval_base_model_results.json" # Tên file riêng biệt cho mốc sàn
MAX_NEW_TOKENS = 200
BATCH_SIZE     = 128

print(f"✅ GPU Vận Hành: {torch.cuda.get_device_name(0)}")

# ------------------------------------------------------------
# 2. Nạp Tokenizer và Mô hình Gốc (Không nạp LoRA Adapter)
# ------------------------------------------------------------
print("📥 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

print("📥 Loading base model thô 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0"
)
model.eval()
print("✅ Mô hình thô đã sẵn sàng!")

# ------------------------------------------------------------
# 3. Đọc dữ liệu tập kiểm thử
# ------------------------------------------------------------
print("📂 Loading test set...")
test_samples = []
with open(TEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        test_samples.append(json.loads(line))

print(f"✅ Tổng {len(test_samples)} mẫu test")

# ------------------------------------------------------------
# 4. Xây dựng Prompt Thô và Chạy Batch Inference
# ------------------------------------------------------------
def build_prompt_tho(sample: dict) -> str:
    # Lấy câu hỏi đơn lẻ, tuyệt đối không bốc thêm ngữ cảnh văn hóa vào prompt
    question = sample.get("standalone_question","")
    
    user_content = f"Hãy trả lời câu hỏi sau về văn hóa Việt Nam:\n[Câu hỏi]: {question}"
    
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def generate_batch(samples: list) -> list[str]:
    all_preds = []
    for start in tqdm(range(0, len(samples), BATCH_SIZE), desc="Evaluating Base Model"):
        batch   = samples[start:start + BATCH_SIZE]
        prompts = [build_prompt_tho(s) for s in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256 # Chiều dài chuỗi ngắn hơn nhiều vì không mang context, chạy cực bốc
        ).to("cuda")

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        input_length = inputs["input_ids"].shape[1]
        for out in output_ids:
            new_tokens = out[input_length:]
            pred = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            all_preds.append(pred)

        del inputs, output_ids
        torch.cuda.empty_cache()

    return all_preds

print("🔥 Bắt đầu sinh câu trả lời tự nhiên từ mô hình thô...")
predictions = generate_batch(test_samples)
references  = [
    s.get("answer_label_explainable") or s.get("answer_label", "")
    for s in test_samples
]

# ------------------------------------------------------------
# 5. Tính toán các chỉ số tự động (ROUGE + BLEU + BERTScore)
# ------------------------------------------------------------
print("\n📊 Tính ROUGE + BLEU...")
rouge = evaluate.load("rouge")
bleu  = evaluate.load("bleu")

rouge_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=False)
bleu_scores = bleu.compute(predictions=predictions, references=[[r] for r in references])

print("\n📊 Tính BERTScore tiếng Việt cho mô hình thô...")
P, R, F1 = bert_score(predictions, references, lang="vi", verbose=True)

print("\n" + "="*50)
print("📈 KẾT QUẢ EVAL MÔ HÌNH THÔ (BASE)")
print("="*50)
print(f"ROUGE-1:  {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2:  {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L:  {rouge_scores['rougeL']:.4f}")
print(f"BLEU:     {bleu_scores['bleu']:.4f}")
print(f"BERTScore F1: {F1.mean().item():.4f}")
print("="*50)

# ------------------------------------------------------------
# 6. Ghi nhật ký chi tiết từng mẫu ra file JSONL
# ------------------------------------------------------------
results_log = []
for i, s in enumerate(test_samples):
    question = s.get("standalone_question","") 
    reference = references[i]
    pred = predictions[i]

    sample_rouge = rouge.compute(predictions=[pred], references=[reference], use_stemmer=False)

    results_log.append({
        "index": i,
        "question": question,
        "prediction": pred,
        "reference": reference,
        "rouge1": round(sample_rouge["rouge1"], 4),
        "rouge2": round(sample_rouge["rouge2"], 4),
        "rougeL": round(sample_rouge["rougeL"], 4),
        "bertscore_f1": round(F1[i].item(), 4)
    })

output = {
    "model": BASE_MODEL_ID,
    "eval_samples": len(test_samples),
    "metrics": {
        "rouge1": round(rouge_scores["rouge1"], 4),
        "rouge2": round(rouge_scores["rouge2"], 4),
        "rougeL": round(rouge_scores["rougeL"], 4),
        "bleu": round(bleu_scores["bleu"], 4),
        "bertscore_precision": round(P.mean().item(), 4),
        "bertscore_recall": round(R.mean().item(), 4),
        "bertscore_f1": round(F1.mean().item(), 4),
    },
    "samples": results_log   
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"\n✅ Đã lưu thành công kết quả mốc sàn tại: {OUTPUT_PATH}")

# In nhanh 3 ví dụ để kiểm tra văn phong thực tế
print("\n📝 VÍ DỤ KẾT QUẢ SỬ TRÍ CỦA NÃO BẢN THÔ:")
for s in results_log[:3]:
    print(f"\n[{s['index']}] ❓ {s['question']}")
    print(f"    🤖 {s['prediction']}")
    print(f"    ✅ {s['reference']}")
    print(f"    📊 ROUGE-L: {s['rougeL']} | BERTScore F1: {s['bertscore_f1']}")

# Finetune QLoRA

In [ ]:
# eval.py — Chạy trên Kaggle GPU T4 x2

import json
import os
import torch
from tqdm import tqdm
import evaluate
import warnings
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ============================================================
# 1. Config
# ============================================================
BASE_MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
LORA_MODEL_ID  = "ohthisischichi/viet-cultural-qa-qwen2.5-lora"
TEST_PATH      = "/kaggle/input/datasets/ohthisischichi/vqa-rag-vietnamculture/final_test.jsonl"
OUTPUT_PATH    = "/kaggle/working/eval_results.json"
MAX_NEW_TOKENS = 200
BATCH_SIZE     = 128

print(f"✅ GPU 0: {torch.cuda.get_device_name(0)}")
print(f"✅ GPU 1: {torch.cuda.get_device_name(1)}")

# ============================================================
# 2. Load model
# ============================================================
print("📥 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)  # ✅ Load từ base model
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

print("📥 Loading base model 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

print("📥 Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, LORA_MODEL_ID)
model.eval()
print("✅ Model sẵn sàng!")

# ============================================================
# 3. Load test set
# ============================================================
print("📂 Loading test set...")
test_samples = []
with open(TEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        test_samples.append(json.loads(line))

print(f"✅ Tổng {len(test_samples)} mẫu test")

# ============================================================
# 4. Batch inference
# ============================================================
def build_prompt(sample: dict) -> str:
    question = sample.get("standalone_question") or sample.get("question", "")
    context  = sample.get("cultural_context", "")
    user_content = (
        f"Dựa vào ngữ cảnh văn hóa được cung cấp, hãy trả lời câu hỏi sau.\n"
        f"[Ngữ cảnh]: {context}\n\n"
        f"[Câu hỏi]: {question}"
    )
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def generate_batch(samples: list) -> list[str]:
    all_preds = []
    for start in tqdm(range(0, len(samples), BATCH_SIZE), desc="Evaluating"):
        batch   = samples[start:start + BATCH_SIZE]
        prompts = [build_prompt(s) for s in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to("cuda")

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        input_length = inputs["input_ids"].shape[1]
        for out in output_ids:
            new_tokens = out[input_length:]
            pred = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            all_preds.append(pred)

        del inputs, output_ids
        torch.cuda.empty_cache()

    return all_preds

print("🔥 Bắt đầu eval...")
predictions = generate_batch(test_samples)
references  = [
    s.get("answer_label_explainable") or s.get("answer_label", "")
    for s in test_samples
]

# ============================================================
# 5. Tính metrics
# ============================================================
print("\n📊 Tính ROUGE + BLEU...")
rouge = evaluate.load("rouge")
bleu  = evaluate.load("bleu")

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=False
)
bleu_scores = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

print("\n" + "="*50)
print("📈 KẾT QUẢ EVAL")
print("="*50)
print(f"ROUGE-1:  {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2:  {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L:  {rouge_scores['rougeL']:.4f}")
print(f"BLEU:     {bleu_scores['bleu']:.4f}")
print("="*50)

# ============================================================
# 6. Lưu kết quả chi tiết ✅
# ============================================================
results_log = []
for i, s in enumerate(test_samples):
    question  = s.get("standalone_question") or s.get("question", "")
    context   = s.get("cultural_context", "")
    reference = references[i]
    pred      = predictions[i]

    # Tính ROUGE-L từng mẫu để dễ quan sát
    sample_rouge = rouge.compute(
        predictions=[pred],
        references=[reference],
        use_stemmer=False
    )

    results_log.append({
        "index":      i,
        "question":   question,
        "context":    context,          # ✅ Full context để quan sát
        "prediction": pred,
        "reference":  reference,
        "rouge1":     round(sample_rouge["rouge1"], 4),
        "rouge2":     round(sample_rouge["rouge2"], 4),
        "rougeL":     round(sample_rouge["rougeL"], 4),
    })

output = {
    "model":        LORA_MODEL_ID,
    "eval_samples": len(test_samples),
    "batch_size":   BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "metrics": {
        "rouge1": round(rouge_scores["rouge1"], 4),
        "rouge2": round(rouge_scores["rouge2"], 4),
        "rougeL": round(rouge_scores["rougeL"], 4),
        "bleu":   round(bleu_scores["bleu"],    4),
    },
    "samples": results_log   # ✅ Lưu toàn bộ, không giới hạn 50
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"\n✅ Đã lưu toàn bộ {len(results_log)} mẫu tại: {OUTPUT_PATH}")

# In vài mẫu kiểm tra
print("\n📝 VÍ DỤ KẾT QUẢ:")
for s in results_log[:3]:
    print(f"\n[{s['index']}] ❓ {s['question']}")
    print(f"    🤖 {s['prediction']}")
    print(f"    ✅ {s['reference']}")
    print(f"    📊 ROUGE-L: {s['rougeL']}")

# RAG

In [ ]:
# eval.py — Chạy trên Kaggle GPU T4 x2 (Đã tích hợp BERTScore đồng bộ)

import json
import os
import torch
from tqdm import tqdm
import evaluate
import warnings
from bert_score import score as bert_score # 🔥 Thêm thư viện bert_score lên đầu

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  # ✅ Đặt TRƯỚC khi import transformers

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ============================================================
# 1. Config
# ============================================================
BASE_MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
LORA_MODEL_ID  = "ohthisischichi/viet-cultural-qa-qwen2.5-lora"
TEST_PATH      = "/kaggle/input/datasets/chichidanghocbai/qa-rag-vietamculture/test_with_rag.jsonl"
OUTPUT_PATH    = "/kaggle/working/eval_with_rag_results.json"
MAX_NEW_TOKENS = 200
BATCH_SIZE     = 128

print(f"✅ GPU 0: {torch.cuda.get_device_name(0)}")
print(f"✅ GPU 1: {torch.cuda.get_device_name(1)}")

# ============================================================
# 2. Load model
# ============================================================
print("📥 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)  # ✅ Load từ base model
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

print("📥 Loading base model 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

print("📥 Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, LORA_MODEL_ID)
model.eval()
print("✅ Model sẵn sàng!")

# ============================================================
# 3. Load test set
# ============================================================
print("📂 Loading test set...")
test_samples = []
with open(TEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        test_samples.append(json.loads(line))

print(f"✅ Tổng {len(test_samples)} mẫu test")

# ============================================================
# 4. Batch inference
# ============================================================
def build_prompt(sample: dict) -> str:
    question = sample.get("standalone_question")
    # evaluation với kết quả truy xuất từ rag
    context  = sample.get("retrieve_context", "")
    user_content = (
        f"Dựa vào ngữ cảnh văn hóa được cung cấp, hãy trả lời câu hỏi sau.\n"
        f"[Ngữ cảnh]: {context}\n\n"
        f"[Câu hỏi]: {question}"
    )
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def generate_batch(samples: list) -> list[str]:
    all_preds = []
    for start in tqdm(range(0, len(samples), BATCH_SIZE), desc="Evaluating"):
        batch   = samples[start:start + BATCH_SIZE]
        prompts = [build_prompt(s) for s in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to("cuda")

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        input_length = inputs["input_ids"].shape[1]
        for out in output_ids:
            new_tokens = out[input_length:]
            pred = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            all_preds.append(pred)

        del inputs, output_ids
        torch.cuda.empty_cache()

    return all_preds

print("🔥 Bắt đầu eval...")
predictions = generate_batch(test_samples)
references  = [
    s.get("answer_label_explainable") or s.get("answer_label", "")
    for s in test_samples
]

# ============================================================
# 5. Tính metrics (ROUGE + BLEU + BERTScore)
# ============================================================
print("\n📊 Tính ROUGE + BLEU...")
rouge = evaluate.load("rouge")
bleu  = evaluate.load("bleu")

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=False
)
bleu_scores = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

print("\n📊 Tính BERTScore cho tiếng Việt...")
# 🔥 Tính BERTScore đồng bộ tại đây, tận dụng GPU để tính toán nhanh
P, R, F1 = bert_score(
    predictions,
    references,
    lang="vi",
    verbose=True
)

print("\n" + "="*50)
print("📈 KẾT QUẢ EVAL")
print("="*50)
print(f"ROUGE-1:  {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2:  {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L:  {rouge_scores['rougeL']:.4f}")
print(f"BLEU:     {bleu_scores['bleu']:.4f}")
print(f"BERTScore F1: {F1.mean().item():.4f}") # In thêm dòng BERTScore
print("="*50)

# ============================================================
# 6. Lưu kết quả chi tiết ✅
# ============================================================
results_log = []
for i, s in enumerate(test_samples):
    question  = s.get("standalone_question","")
    retrieve_context   = s.get("retrieve_context", "")
    real_context = s.get("cultural_context", "")
    reference = references[i]
    pred      = predictions[i]

    # Tính ROUGE-L từng mẫu để dễ quan sát
    sample_rouge = rouge.compute(
        predictions=[pred],
        references=[reference],
        use_stemmer=False
    )

    results_log.append({
        "index":      i,
        "question":   question,
        "real_cultural_context": real_context,
        "retrive_context": retrieve_context,
        "prediction": pred,
        "reference":  reference,
        "rouge1":     round(sample_rouge["rouge1"], 4),
        "rouge2":     round(sample_rouge["rouge2"], 4),
        "rougeL":     round(sample_rouge["rougeL"], 4),
        "bertscore_f1": round(F1[i].item(), 4) # 🔥 Ép chỉ số BERTScore của từng mẫu lẻ vào log luôn
    })

output = {
    "model":        LORA_MODEL_ID,
    "eval_samples": len(test_samples),
    "batch_size":   BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "metrics": {
        "rouge1": round(rouge_scores["rouge1"], 4),
        "rouge2": round(rouge_scores["rouge2"], 4),
        "rougeL": round(rouge_scores["rougeL"], 4),
        "bleu":   round(bleu_scores["bleu"],    4),
        "bertscore_precision": round(P.mean().item(), 4), # Thêm đủ bộ 3 chỉ số BERTScore toàn cục
        "bertscore_recall": round(R.mean().item(), 4),
        "bertscore_f1": round(F1.mean().item(), 4),
    },
    "samples": results_log   
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"\n✅ Đã lưu toàn bộ {len(results_log)} mẫu và đầy đủ metrics tại: {OUTPUT_PATH}")

# In vài mẫu kiểm tra văn phong sinh chữ
print("\n📝 VÍ DỤ KẾT QUẢ:")
for s in results_log[:3]:
    print(f"\n[{s['index']}] ❓ {s['question']}")
    print(f"    🤖 {s['prediction']}")
    print(f"    ✅ {s['reference']}")
    print(f"    📊 ROUGE-L: {s['rougeL']} | BERTScore F1: {s['bertscore_f1']}")

In [5]:
# Phần này sẽ so sánh metric của 3 mô hình: Base Model, Mô hình có LoRA, Mô hình có LoRA + RAG.
# Kết quả sẽ được tổng hợp thành bảng để dễ so sánh và phân tích.
import json
import pandas as pd
import os
BASE_MODEL_PATH = "../data/evaluation/eval_base_model_results.json"
LORA_MODEL_PATH = "../data/evaluation/eval_results_with_bert.json"
RAG_MODEL_PATH  = "../data/evaluation/eval_with_rag_results.json"

def load_eval_results(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return {
        "model": data.get("model", "unknown"),
        "rouge1": data.get("metrics", {}).get("rouge1", 0),
        "rouge2": data.get("metrics", {}).get("rouge2", 0),
        "rougeL": data.get("metrics", {}).get("rougeL", 0),
        "bleu": data.get("metrics", {}).get("bleu", 0),
        "bertscore_f1": data.get("metrics", {}).get("bertscore_f1", 0)
    }

base_results = load_eval_results(BASE_MODEL_PATH)
lora_results = load_eval_results(LORA_MODEL_PATH)
rag_results  = load_eval_results(RAG_MODEL_PATH)
df = pd.DataFrame([base_results, lora_results, rag_results])
print("\n📊 BẢNG SO SÁNH KẾT QUẢ EVAL")
print(df.to_string(index=False))



📊 BẢNG SO SÁNH KẾT QUẢ EVAL
                                             model  rouge1  rouge2  rougeL   bleu  bertscore_f1
                          Qwen/Qwen2.5-3B-Instruct  0.6101  0.2945  0.3377 0.0700        0.7204
      ohthisischichi/viet-cultural-qa-qwen2.5-lora  0.7408  0.4367  0.4659 0.2476        0.8006
ohthisischichi/viet-cultural-qa-qwen2.5-lora + RAG  0.7247  0.4058  0.4444 0.2204        0.7898
